# check_c.py on Kaggle

Does the label-free score C rank test images by how much depth helps? Takes about 8 minutes.

**Before running**
1. Add Input: your `rgbx-data` dataset (data zip) and a `rgbx-weights` dataset holding the two folders `full_rgbd_s0` and `full_rgb_s0` (each with `final.pt`, from `MyDrive/rgbx/runs/`). Zips are fine.
2. Settings: Accelerator **GPU T4 x2**, Internet **on**.
3. Run All, then paste the 6 lines printed at the end of the last cell back into the project thread.

In [ ]:
# Cell 1: check the GPU, get the code
import os, glob, subprocess, torch
def sh(cmd):
    print('$', cmd, flush=True); return subprocess.run(cmd, shell=True).returncode

print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Turn on the GPU: Settings > Accelerator > GPU T4 x2'
torch.ones(1).cuda().sum().item()   # an error here means this GPU is not supported: choose GPU T4 x2

CODE = '/tmp/code'
sh(f'rm -rf {CODE} && git clone -q https://github.com/talha-ansarii/btech-project.git {CODE}')
sh(f'cd {CODE} && git log --oneline -1')
sh(f'pip -q install -r {CODE}/requirements.txt')

In [ ]:
# Cell 2: find the data and the two full-data model weights
INPUT = '/kaggle/input'
def find(pattern):
    hits = glob.glob(f'{INPUT}/**/{pattern}', recursive=True) + glob.glob(f'/tmp/unz/**/{pattern}', recursive=True)
    if not hits:   # Kaggle sometimes keeps uploaded zips as zips: unpack them once and look again
        for z in glob.glob(f'{INPUT}/**/*.zip', recursive=True):
            sh(f'mkdir -p /tmp/unz && unzip -q -o "{z}" -d /tmp/unz')
        hits = glob.glob(f'/tmp/unz/**/{pattern}', recursive=True)
    assert hits, f'{pattern} not found: add the dataset that holds it with "Add Input"'
    return os.path.dirname(hits[0])

DATA = find('meta.json')
RGBD_RUN = find('full_rgbd_s0/final.pt')   # folder that holds final.pt of the RGB-D model
RGB_RUN = find('full_rgb_s0/final.pt')
RUNS = '/tmp/runs'                          # check_c.py wants <runs>/full_rgbd_s0/final.pt
sh(f'rm -rf {RUNS} && mkdir -p {RUNS} && ln -s {RGBD_RUN} {RUNS}/full_rgbd_s0 && ln -s {RGB_RUN} {RUNS}/full_rgb_s0')
CACHE = '/kaggle/working/cache'
print(len(os.listdir(f'{DATA}/train/rgb')), len(os.listdir(f'{DATA}/test/rgb')), '(expect 795 654)')
sh(f'ls -la {RUNS}/full_rgbd_s0/ {RUNS}/full_rgb_s0/')

In [ ]:
# Cell 3: C for the training pool (thresholds only, ~1 min), then the check on the 654 test images (~5 min)
sh(f'python {CODE}/complementarity.py --data {DATA} --split train --out {CACHE}/labelfree_train.npz --no-features')
sh(f'python {CODE}/check_c.py --data {DATA} --runs {RUNS} --cache {CACHE}')